In [9]:
#1.1
import pandas as pd
#1.2
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
#1.3
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
#1.4
import joblib
from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns



## 1.1 Xử lý dữ liệu trùng lặp, rỗng

In [2]:
def load_and_clean_data(file_path):
    """
    Nạp và làm sạch dữ liệu từ tệp spam.csv
    Đầu vào: Đường dẫn tệp tin
    Đầu ra: DataFrame chứa 2 cột ['Category', 'Message'] đã sạch nhiễu
    """
    try:
        # Bước 1: Nạp dữ liệu
        df = pd.read_csv(file_path, encoding='latin-1')
        
        # Bước 2: Xử lý nhiễu
        df.drop_duplicates(inplace=True)
        df.dropna(inplace=True)
        
        return df
        
    except FileNotFoundError:
        print("Lỗi: Không tìm thấy tệp CSV. Hãy kiểm tra lại đường dẫn.")
        return None


if __name__ == "__main__":
    
    df_clean = load_and_clean_data('spam.csv')
    
    if df_clean is not None:
        print("Dữ liệu đã được làm sạch thành công!")
        print(f"Kích thước DataFrame hiện tại: {df_clean.shape}")
        print("\n5 dòng dữ liệu đầu tiên:")
        print(df_clean.head())

Dữ liệu đã được làm sạch thành công!
Kích thước DataFrame hiện tại: (5157, 2)

5 dòng dữ liệu đầu tiên:
  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...


## 1.2 Tiền xử lý ngôn ngữ

In [3]:
# 1. Tải các kho ngữ liệu (corpus) và mô hình ngôn ngữ của NLTK
# Tham số quiet=True để ẩn các dòng log tải xuống trên terminal
nltk.download('punkt', quiet=True)
# mô hình ML thực hiện phân tách câu 
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True) 
# danh sách các từ dừng
nltk.download('wordnet', quiet=True)   
# cơ sở dữ liệu từ vựng TA khổng lồ để WordNetLemmatizer() tra cứu

# 2. Khởi tạo hằng số toàn cục (Global Constants)
# Ép kiểu danh sách từ dừng sang cấu trúc tập hợp (Set)
STOP_WORDS = set(stopwords.words('english'))

# Khởi tạo đối tượng xử lý hình thái học
LEMMATIZER = WordNetLemmatizer()
#hàm này tra cứu từ điểm và phân tích ngữ pháp để tìm từ gốc (ví dụ: better ->good)

In [4]:
def clean_text(text):
    # 1. Ép kiểu về string và chuyển chữ thường (Lowercasing)
    text = str(text).lower()

    # 2. Xóa các đường dẫn URL (http, https, www)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 3. Xóa các địa chỉ email (ví dụ: abc@gmail.com)
    text = re.sub(r'\S+@\S+', '', text)

    # 4. Xóa các ký tự không phải là chữ cái hoặc số (Ký tự đặc biệt và dấu câu)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    """
    re.sub(pattern, repl, string, count=0, flags=0)
    pattern: biểu thức chính quy
    repl: chuỗi thya thế
    string: chuỗi gốc cần xử lý
    """

    # 5. Tách từ (Tokenization)
    # Sử dụng mô hình Punkt đã tải để chia chuỗi thành danh sách các từ khi gọi hàm word_tokenize()
    tokens = word_tokenize(text)

    # 6. Loại bỏ từ dừng (Stopwords)
    # Kiểm tra từng từ, nếu không nằm trong tập STOP_WORDS thì mới giữ lại
    tokens = [word for word in tokens if word not in STOP_WORDS]

    # 7. Đưa từ về nguyên thể (Lemmatization)
    # Sử dụng đối tượng LEMMATIZER để ánh xạ từ về dạng gốc có nghĩa
    tokens = [LEMMATIZER.lemmatize(word) for word in tokens]

    # 8. Tái cấu trúc (Reconstruction)
    # Nối các tokens đã xử lý lại thành một chuỗi văn bản hoàn chỉnh
    return " ".join(tokens)

    #-----------------#

if __name__ == "__main__":
    
    if df_clean is not None:

        print("Đang thực hiện tiền xử lý NLP cho toàn bộ dữ liệu...")
        
        # Bước 2: Xử lý NLP (Giai đoạn 2)
        df_clean['Clean_Message'] = df_clean['Message'].apply(clean_text)
        
        # Kiểm tra kết quả đầu ra 
        print("\nHoàn tất Giai đoạn 2. Kết cấu dữ liệu hiện tại:")
        print(df_clean[['Category', 'Message', 'Clean_Message']].head())


Đang thực hiện tiền xử lý NLP cho toàn bộ dữ liệu...

Hoàn tất Giai đoạn 2. Kết cấu dữ liệu hiện tại:
  Category                                            Message  \
0      ham  Go until jurong point, crazy.. Available only ...   
1      ham                      Ok lar... Joking wif u oni...   
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...   
3      ham  U dun say so early hor... U c already then say...   
4      ham  Nah I don't think he goes to usf, he lives aro...   

                                       Clean_Message  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry 2 wkly comp win fa cup final tkts 2...  
3                u dun say early hor u c already say  
4           nah dont think go usf life around though  


In [ ]:
#df_clean.to_csv('spam_test.csv')
#test thử data

## 1.3 Khởi tạo K - Fold và TF - IDF

In [11]:
# 1. Khởi tạo cơ chế phân tách Stratified K-Fold (K=5 theo kế hoạch)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
"""
random_state =  giá trị để để hàm shuffle tráo ra cùng một bộ dữu liệu (có thể mang giá trị nào cũng được)
"""
# 2. Khởi tạo đối tượng lượng hóa ngôn ngữ TF-IDF
# Giới hạn max_features=5000 để loại bỏ các từ quá hiếm, giúp tránh bùng nổ RAM
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
"""
max_features lọc ra 5000 từ có tần suất (tf) cao nhất trước khi tính điểm tf-idf
theo định lý zipf, tần suất của 1 từ = 1/r với r = rank của chúng, nên ta lấy 5000 giá trị đầu 
overfitting nếu lấy quá nhiều? 
"""
# Khai báo trước mảng X (Đặc trưng - Nội dung) và y (Nhãn - Spam/Ham)
# Quy ước trong Machine Learning: X viết hoa (Ma trận/Vector), y viết thường (Vector 1 chiều)
X_text = df_clean['Clean_Message'].values
y_labels = df_clean['Category'].values

## 1.4 Thực hiện F - Fold, TF - IDF, SMOTE

In [7]:
# Khởi tạo thuật toán SMOTE 
smote = SMOTE(random_state=42)

fold_no = 1
# skf.split() sẽ trả về danh sách các vị trí (index) để cắt dữ liệu
for train_index, test_index in skf.split(X_text, y_labels):
    print(f"\n--- Đang xử lý Fold thứ {fold_no} ---")

    # 1. Cắt dữ liệu thành tập Huấn luyện (Train) và Kiểm tra (Test)
    X_train, X_test = X_text[train_index], X_text[test_index]
    y_train, y_test = y_labels[train_index], y_labels[test_index]

    # Lượng hóa TF-IDF
    # fit_transform cho Train và chỉ transform cho Test
    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
    X_test_tfidf = tfidf_vectorizer.transform(X_test)

    # Cân bằng dữ liệu (SMOTE)
    # Chỉ áp dụng SMOTE lên tập Train
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_tfidf, y_train)

    print(f"Số email Train ban đầu: {X_train_tfidf.shape[0]} | Sau SMOTE: {X_train_resampled.shape[0]}")
    fold_no += 1

# 4. Lưu lại đối tượng từ điển TF-IDF cuối cùng để dùng cho Giao diện Web
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
print("\nĐã xuất tệp tfidf_vectorizer.pkl thành công!")


--- Đang xử lý Fold thứ 1 ---
Số email Train ban đầu: 4125 | Sau SMOTE: 7224

--- Đang xử lý Fold thứ 2 ---
Số email Train ban đầu: 4125 | Sau SMOTE: 7226

--- Đang xử lý Fold thứ 3 ---
Số email Train ban đầu: 4126 | Sau SMOTE: 7226

--- Đang xử lý Fold thứ 4 ---
Số email Train ban đầu: 4126 | Sau SMOTE: 7226

--- Đang xử lý Fold thứ 5 ---
Số email Train ban đầu: 4126 | Sau SMOTE: 7226

Đã xuất tệp tfidf_vectorizer.pkl thành công!
